In [2]:
"""
USDINR LightGBM + HMM Experiment
=================================
Compares three feature sets on 60s and 300s return forecasting:
  - BASE46       : 46 microstructure features
  - CHALLENGER50 : BASE46 + 4 challenger features
  - CHALLENGER54 : CHALLENGER50 + 4 HMM soft state probabilities

Split  : Days 1-15 train | Days 16-20 val | Days 21-25 test
Outputs: results/lgbm_hmm/metrics.json
         results/lgbm_hmm/comparison.csv
         results/lgbm_hmm/feature_importance.csv
         results/lgbm_hmm/daily_ic.csv
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import BASE46, CHALLENGER50, CHALLENGERS4
from data.processing.usdinr.regimes import (
    add_soft_hmm_features,
    save_hmm_model,
    HMM_PROB_COLS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\lgbm_hmm")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS   = [60, 300]
TRAIN_DAYS = 15
VAL_DAYS   = 5

LGB_PARAMS = {
    "objective":             "regression",
    "metric":                "rmse",
    "verbosity":             -1,
    "boosting_type":         "gbdt",
    "num_leaves":            31,
    "learning_rate":         0.05,
    "feature_fraction":      0.8,
    "bagging_fraction":      0.8,
    "bagging_freq":          5,
    "min_child_samples":     100,
    "n_estimators":          1000,
    "early_stopping_rounds": 50,
}

# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

contracts  = ["26OCTFUT", "26NOVFUT"]
all_frames = []

for contract in contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df_tmp           = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_tmp["symbol"] = sym
            all_frames.append(df_tmp)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded — check S3 paths and contracts")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build 50 microstructure features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building microstructure features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Features: %d rows, %d columns", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Add tick_count (needed by HMM)
# ---------------------------------------------------------------------------

log.info("=== Step 3: Adding tick_count ===")

# tick_count is already carried through by USDINRFeatureBuilder from raw data.
# Synthetic fill rows (regularized seconds with no events) have tick_count=0.
# Just validate it is present and clean.
if "tick_count" not in df.columns:
    raise RuntimeError(
        "tick_count column not found in builder output. "
        "Check that the raw parquet files include a tick_count column."
    )

df["tick_count"] = df["tick_count"].fillna(0).astype(int)
log.info("tick_count present | mean=%.2f  max=%d  zeros=%.1f%%",
         df["tick_count"].mean(),
         df["tick_count"].max(),
         100 * (df["tick_count"] == 0).mean())
log.info("tick_count added | mean=%.2f  max=%d",
         df["tick_count"].mean(), df["tick_count"].max())

# ---------------------------------------------------------------------------
# Step 4 — Train/val/test split dates
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting by date ===")

df         = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
all_dates  = sorted(df["date"].unique())
n_days     = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(
        f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}"
    )

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])

log.info("Train: %d days (through %s)  Val: %d days  Test: %d days",
         len(train_dates), train_end, len(val_dates), len(test_dates))

# ---------------------------------------------------------------------------
# Step 5 — Fit HMM on training data, add soft probs to all rows
# ---------------------------------------------------------------------------

log.info("=== Step 5: Fitting HMM ===")

df, hmm_model = add_soft_hmm_features(
    df             = df,
    train_end_date = train_end,
    K              = 4,
    restarts       = 3,
    max_iter       = 70,
    tol            = 1e-6,
    seed           = 42,
)

save_hmm_model(hmm_model, MODEL_DIR / "hmm_usdinr.pkl")

log.info("HMM soft probs added: %s", HMM_PROB_COLS)
log.info("State distribution (all rows):\n%s",
         df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Step 6 — Build return targets
# ---------------------------------------------------------------------------

log.info("=== Step 6: Building return targets ===")

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
for h in HORIZONS:
    df[f"ret_{h}s"] = (g["log_mid"].shift(-h) - df["log_mid"]) * 1e4
    log.info("Target ret_%ds: %.1f%% non-null",
             h, df[f"ret_{h}s"].notna().mean() * 100)

# ---------------------------------------------------------------------------
# Step 7 — Define feature sets
# ---------------------------------------------------------------------------

CHALLENGER54 = CHALLENGER50 + HMM_PROB_COLS
assert len(CHALLENGER54) == 54, f"Expected 54 features, got {len(CHALLENGER54)}"

FEATURE_SETS = {
    "base46":        BASE46,
    "challenger50":  CHALLENGER50,
    "challenger54":  CHALLENGER54,
}

# ---------------------------------------------------------------------------
# Step 8 — Train and evaluate
# ---------------------------------------------------------------------------

log.info("=== Step 8: Training models ===")

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

all_metrics     = {}
all_importance  = {}
comparison_rows = []

oof         = df[["symbol", "ts_sec", "date"]].copy()
oof["split"] = np.where(mask_val, "val", np.where(mask_test, "test", "train"))


def compute_metrics(y_true, y_pred, label):
    rmse      = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    ic, _     = spearmanr(y_true, y_pred)
    hit       = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    log.info("%s | IC=%+.4f  RMSE=%.4f  Hit=%.4f  n=%d",
             label, ic, rmse, hit, len(y_true))
    return {"ic": ic, "rmse": rmse, "hit_rate": hit, "n": len(y_true)}


for feat_name, features in FEATURE_SETS.items():
    log.info("--- Feature set: %s (%d features) ---", feat_name, len(features))

    for horizon in HORIZONS:
        target  = f"ret_{horizon}s"
        run_key = f"{feat_name}/{target}"

        valid = df[features + [target]].notna().all(axis=1)

        tr = df[mask_train & valid]
        va = df[mask_val   & valid]
        te = df[mask_test  & valid]

        log.info("%s | train=%d  val=%d  test=%d", run_key, len(tr), len(va), len(te))

        X_tr, y_tr = tr[features].values, tr[target].values
        X_va, y_va = va[features].values, va[target].values
        X_te, y_te = te[features].values, te[target].values

        ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
        ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

        params = {k: v for k, v in LGB_PARAMS.items()
                  if k not in ("n_estimators", "early_stopping_rounds")}

        model = lgb.train(
            params,
            ds_tr,
            num_boost_round = LGB_PARAMS["n_estimators"],
            valid_sets      = [ds_va],
            callbacks       = [
                lgb.early_stopping(LGB_PARAMS["early_stopping_rounds"], verbose=False),
                lgb.log_evaluation(200),
            ],
        )

        pred_tr = model.predict(X_tr)
        pred_va = model.predict(X_va)
        pred_te = model.predict(X_te)

        metrics = {
            "train": compute_metrics(y_tr, pred_tr, f"{run_key}/train"),
            "val":   compute_metrics(y_va, pred_va, f"{run_key}/val"),
            "test":  compute_metrics(y_te, pred_te, f"{run_key}/test"),
            "best_iteration": model.best_iteration,
        }
        all_metrics[run_key] = metrics

        # Feature importance
        imp = pd.DataFrame({
            "feature":        features,
            "gain":           model.feature_importance("gain"),
            "split_count":    model.feature_importance("split"),
            "feat_set":       feat_name,
            "horizon_s":      horizon,
            "is_hmm":         [f in HMM_PROB_COLS for f in features],
            "is_challenger":  [f in CHALLENGERS4   for f in features],
        }).sort_values("gain", ascending=False)
        all_importance[run_key] = imp

        # OOF predictions
        pred_col   = f"pred_{feat_name}_{target}"
        actual_col = f"actual_{target}"
        oof.loc[va.index, pred_col]   = pred_va
        oof.loc[te.index, pred_col]   = pred_te
        oof.loc[va.index, actual_col] = y_va
        oof.loc[te.index, actual_col] = y_te

        comparison_rows.append({
            "feat_set":   feat_name,
            "n_features": len(features),
            "horizon_s":  horizon,
            "train_ic":   round(metrics["train"]["ic"], 4),
            "val_ic":     round(metrics["val"]["ic"],   4),
            "test_ic":    round(metrics["test"]["ic"],  4),
            "test_rmse":  round(metrics["test"]["rmse"], 4),
            "test_hit":   round(metrics["test"]["hit_rate"], 4),
            "best_iter":  model.best_iteration,
        })

# ---------------------------------------------------------------------------
# Step 9 — HMM feature contribution analysis
# ---------------------------------------------------------------------------

log.info("=== Step 9: HMM feature contribution ===")

for horizon in HORIZONS:
    b50_ic  = all_metrics[f"challenger50/ret_{horizon}s"]["test"]["ic"]
    b54_ic  = all_metrics[f"challenger54/ret_{horizon}s"]["test"]["ic"]
    b46_ic  = all_metrics[f"base46/ret_{horizon}s"]["test"]["ic"]
    delta   = b54_ic - b50_ic
    pct     = delta / abs(b50_ic) * 100 if b50_ic != 0 else 0
    log.info(
        "ret_%ds | BASE46=%.4f  CHALLENGER50=%.4f  CHALLENGER54=%.4f  "
        "HMM delta=%+.4f (%+.1f%% vs C50)",
        horizon, b46_ic, b50_ic, b54_ic, delta, pct,
    )

    # Rank of HMM features within CHALLENGER54 model
    imp54 = all_importance[f"challenger54/ret_{horizon}s"].reset_index(drop=True)
    imp54["rank"] = imp54.index + 1
    hmm_imp = imp54[imp54["is_hmm"]][["feature", "gain", "rank"]]
    log.info("HMM feature ranks (ret_%ds):\n%s", horizon, hmm_imp.to_string(index=False))

# ---------------------------------------------------------------------------
# Step 10 — Daily IC breakdown
# ---------------------------------------------------------------------------

log.info("=== Step 10: Daily IC breakdown ===")

daily_ic_rows = []
eval_mask     = mask_val | mask_test

for feat_name in FEATURE_SETS:
    for horizon in HORIZONS:
        target   = f"ret_{horizon}s"
        pred_col = f"pred_{feat_name}_{target}"
        act_col  = f"actual_{target}"

        if pred_col not in oof.columns:
            continue

        sub = oof[eval_mask & oof[pred_col].notna() & oof[act_col].notna()].copy()

        for day, grp in sub.groupby("date"):
            if len(grp) < 50:
                continue
            ic, _ = spearmanr(grp[act_col], grp[pred_col])
            daily_ic_rows.append({
                "date":      str(day),
                "feat_set":  feat_name,
                "horizon_s": horizon,
                "ic":        round(float(ic), 4),
                "n":         len(grp),
                "split":     "val" if day in val_dates else "test",
            })

daily_ic = pd.DataFrame(daily_ic_rows)

# ---------------------------------------------------------------------------
# Step 11 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 11: Saving outputs ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({
        "split_dates": {
            "train": [str(d) for d in train_dates],
            "val":   [str(d) for d in val_dates],
            "test":  [str(d) for d in test_dates],
        },
        "hmm_train_end": train_end,
        "lgb_params":    LGB_PARAMS,
        "metrics":       all_metrics,
    }, f, indent=2)
log.info("Saved metrics.json")

imp_df = pd.concat(all_importance.values(), ignore_index=True)
imp_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)
log.info("Saved feature_importance.csv")

oof[eval_mask].to_csv(OUT_DIR / "oof_predictions.csv", index=False)
log.info("Saved oof_predictions.csv")

if not daily_ic.empty:
    daily_ic.to_csv(OUT_DIR / "daily_ic.csv", index=False)
    log.info("Saved daily_ic.csv")

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
log.info("Saved comparison.csv")

# ---------------------------------------------------------------------------
# Step 12 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("BASE46  vs  CHALLENGER50  vs  CHALLENGER54  —  COMPARISON SUMMARY")
print("=" * 75)
print(comparison.to_string(index=False))

print("\n--- IC delta vs CHALLENGER50 ---")
for horizon in HORIZONS:
    c50 = comparison[
        (comparison["feat_set"] == "challenger50") &
        (comparison["horizon_s"] == horizon)
    ]["test_ic"].values[0]
    c54 = comparison[
        (comparison["feat_set"] == "challenger54") &
        (comparison["horizon_s"] == horizon)
    ]["test_ic"].values[0]
    b46 = comparison[
        (comparison["feat_set"] == "base46") &
        (comparison["horizon_s"] == horizon)
    ]["test_ic"].values[0]
    print(f"  ret_{horizon}s:  BASE46={b46:+.4f}  "
          f"CHALLENGER50={c50:+.4f}  CHALLENGER54={c54:+.4f}  "
          f"HMM_delta={c54-c50:+.4f}")

if not daily_ic.empty:
    print("\n--- Daily IC mean ± std (test days only) ---")
    print(
        daily_ic[daily_ic["split"] == "test"]
        .groupby(["feat_set", "horizon_s"])["ic"]
        .agg(["mean", "std", "min"])
        .round(4)
        .to_string()
    )

print("=" * 75)

11:43:07  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

11:44:38  INFO  Raw: 420432 rows | symbols: ['USDINR26OCTFUT', 'USDINR26NOVFUT']
11:44:38  INFO  === Step 2: Building microstructure features ===
11:44:42  INFO  preprocess: 759442 rows after regularization
11:44:42  INFO  calculate_features: building CORE31 ...
11:45:01  INFO  build_core31_from_regularized: built 759442 rows, 31 CORE31 features
11:45:01  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
11:45:06  INFO  _build_book_geometry_features: built 759442 rows
11:45:06  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
11:45:08  INFO  _build_shock_recovery_features: bid_shocks=19608 ask_shocks=37112
11:45:08  INFO  _build_shock_recovery_features: built 759442 rows
11:45:08  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
11:45:10  INFO  _build_update_clustering_features: obs=420432 book_change=285978 bid=128313 ask=206034
11:45:11  INFO  _build_update_clustering_features: built 759442 rows
11:45:11  INFO  cal


BASE46  vs  CHALLENGER50  vs  CHALLENGER54  —  COMPARISON SUMMARY
    feat_set  n_features  horizon_s  train_ic  val_ic  test_ic  test_rmse  test_hit  best_iter
      base46          46         60    0.3381  0.2300   0.2251     1.1253    0.4766         40
      base46          46        300    0.3268  0.1688   0.1240     2.6821    0.4740          4
challenger50          50         60    0.3278  0.2293   0.2333     1.2246    0.4984         21
challenger50          50        300    0.4210  0.1351   0.0993     2.8693    0.5024         12
challenger54          54         60    0.3164  0.2179   0.2298     1.2257    0.4981         18
challenger54          54        300    0.4241  0.1512   0.0919     2.8675    0.5110         10

--- IC delta vs CHALLENGER50 ---
  ret_60s:  BASE46=+0.2251  CHALLENGER50=+0.2333  CHALLENGER54=+0.2298  HMM_delta=-0.0035
  ret_300s:  BASE46=+0.1240  CHALLENGER50=+0.0993  CHALLENGER54=+0.0919  HMM_delta=-0.0074

--- Daily IC mean ± std (test days only) ---
       

In [1]:
"""
USDINR TCN Experiment — Temporal Convolutional Network
=======================================================
Compares TCN against LightGBM baseline on return forecasting.

Architecture:
  Input  : [batch, seq_len=60, n_features=50]  (60-second lookback window)
  TCN    : 5 dilated causal conv blocks (dilation 1,2,4,8,16)
  Output : ret_60s  AND  ret_300s  (dual-head)

Split   : Days 1-15 train | Days 16-20 val | Days 21-25 test
Metric  : Spearman IC (primary), RMSE (secondary)
Outputs : results/tcn/metrics.json
          results/tcn/comparison.csv   (TCN vs LightGBM)
          results/tcn/daily_ic.csv
          results/tcn/oof_predictions.csv
          models/tcn_usdinr.pt

No plots — CSV/JSON outputs only.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import BASE46, CHALLENGER50

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\tcn")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SET  = CHALLENGER50          # 50 features — best LightGBM set
SEQ_LEN      = 60                    # 60-second lookback window
HORIZONS     = [60, 300]
TRAIN_DAYS   = 15
VAL_DAYS     = 5

# TCN architecture
TCN_CHANNELS  = 64                   # channels per conv block
TCN_LAYERS    = 5                    # dilation = 2^(0..4) = 1,2,4,8,16
KERNEL_SIZE   = 3
DROPOUT       = 0.2

# Training
BATCH_SIZE    = 512
MAX_EPOCHS    = 100
PATIENCE      = 10                   # early stopping on val IC
LR            = 1e-3
WEIGHT_DECAY  = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log.info("Device: %s", DEVICE)

# ---------------------------------------------------------------------------
# TCN Model
# ---------------------------------------------------------------------------

class CausalConvBlock(nn.Module):
    """Single dilated causal conv block with residual connection."""

    def __init__(self, in_ch: int, out_ch: int, kernel: int, dilation: int, dropout: float):
        super().__init__()
        # Causal padding: pad left only so output doesn't see future
        pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(
            in_ch, out_ch, kernel,
            padding=pad, dilation=dilation
        )
        self.pad   = pad
        self.norm  = nn.LayerNorm(out_ch)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)
        # Residual projection if channels differ
        self.proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: [B, C, T]
        out = self.conv(x)
        out = out[:, :, :-self.pad] if self.pad > 0 else out  # remove future leak
        # LayerNorm expects [B, T, C]
        out = self.norm(out.transpose(1, 2)).transpose(1, 2)
        out = self.act(out)
        out = self.drop(out)
        return out + self.proj(x)


class TCN(nn.Module):
    """
    Temporal Convolutional Network for HFT return forecasting.
    Input  : [B, T, F]  (batch, seq_len, n_features)
    Output : [B, 2]     (ret_60s prediction, ret_300s prediction)
    """

    def __init__(
        self,
        n_features: int,
        n_channels: int = 64,
        n_layers: int   = 5,
        kernel: int     = 3,
        dropout: float  = 0.2,
    ):
        super().__init__()

        # Input projection: F → n_channels
        self.input_proj = nn.Linear(n_features, n_channels)

        # TCN blocks with exponentially increasing dilation
        blocks = []
        for i in range(n_layers):
            dilation = 2 ** i
            blocks.append(
                CausalConvBlock(n_channels, n_channels, kernel, dilation, dropout)
            )
        self.blocks = nn.ModuleList(blocks)

        # Dual output heads — one per horizon
        self.head_60s  = nn.Linear(n_channels, 1)
        self.head_300s = nn.Linear(n_channels, 1)

    def forward(self, x):
        # x: [B, T, F]
        x = self.input_proj(x)          # [B, T, C]
        x = x.transpose(1, 2)           # [B, C, T]  (Conv1d format)
        for block in self.blocks:
            x = block(x)                # [B, C, T]
        last = x[:, :, -1]              # [B, C]  — last timestep only
        return self.head_60s(last).squeeze(-1), self.head_300s(last).squeeze(-1)


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class SequenceDataset(Dataset):
    """
    Builds non-overlapping* sliding windows over a single day's data.
    *Windows from adjacent days do NOT overlap — day boundary respected.

    Each sample:
        X : [seq_len, n_features]  — standardized feature lookback
        y60  : scalar float        — ret_60s target
        y300 : scalar float        — ret_300s target
        idx  : int                 — row index in original df
    """

    def __init__(
        self,
        df: pd.DataFrame,
        features: list,
        seq_len: int,
        scaler: StandardScaler = None,
        fit_scaler: bool = False,
    ):
        self.seq_len  = seq_len
        self.features = features
        self.samples  = []  # (X, y60, y300, orig_idx)

        # Fit or apply scaler
        feat_vals = df[features].values.astype(np.float32)
        if fit_scaler:
            self.scaler = StandardScaler()
            feat_vals   = self.scaler.fit_transform(feat_vals)
        else:
            self.scaler = scaler
            feat_vals   = scaler.transform(feat_vals)

        # Store back for window extraction
        df = df.copy()
        df[features] = feat_vals

        # Build windows per day — no cross-day leakage
        for day, grp in df.groupby("date", sort=True):
            grp = grp.sort_values("ts_sec").reset_index(drop=True)
            n   = len(grp)

            feat = grp[features].values.astype(np.float32)
            y60  = grp["ret_60s"].values.astype(np.float32)
            y300 = grp["ret_300s"].values.astype(np.float32)
            orig = grp.index.values

            # Slide window: start from seq_len so we have full history
            for i in range(seq_len, n):
                # Only include rows where both targets are valid
                if np.isnan(y60[i]) or np.isnan(y300[i]):
                    continue
                # Check feature window has no NaN
                window = feat[i - seq_len : i]
                if np.isnan(window).any():
                    continue

                self.samples.append((window, y60[i], y300[i], orig[i]))

        log.info(
            "SequenceDataset: %d samples from %d days (seq_len=%d)",
            len(self.samples),
            df["date"].nunique(),
            seq_len,
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        X, y60, y300, orig_idx = self.samples[idx]
        return (
            torch.from_numpy(X),
            torch.tensor(y60,  dtype=torch.float32),
            torch.tensor(y300, dtype=torch.float32),
            orig_idx,
        )


# ---------------------------------------------------------------------------
# Training utilities
# ---------------------------------------------------------------------------

def compute_ic(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Spearman rank IC — primary metric."""
    if len(y_true) < 10:
        return 0.0
    ic, _ = spearmanr(y_true, y_pred)
    return float(ic) if not np.isnan(ic) else 0.0


def run_epoch(model, loader, optimizer, criterion, device, train: bool):
    model.train(train)
    total_loss = 0.0
    preds_60, preds_300 = [], []
    trues_60, trues_300 = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y60, y300, _ in loader:
            X   = X.to(device)
            y60 = y60.to(device)
            y300 = y300.to(device)

            p60, p300 = model(X)
            loss = criterion(p60, y60) + criterion(p300, y300)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(X)
            preds_60.extend(p60.detach().cpu().numpy())
            preds_300.extend(p300.detach().cpu().numpy())
            trues_60.extend(y60.detach().cpu().numpy())
            trues_300.extend(y300.detach().cpu().numpy())

    n = max(len(preds_60), 1)
    ic60  = compute_ic(np.array(trues_60),  np.array(preds_60))
    ic300 = compute_ic(np.array(trues_300), np.array(preds_300))
    return total_loss / n, ic60, ic300


def predict_all(model, loader, device):
    model.eval()
    preds_60, preds_300, orig_idxs = [], [], []
    with torch.no_grad():
        for X, _, _, orig_idx in loader:
            X = X.to(device)
            p60, p300 = model(X)
            preds_60.extend(p60.cpu().numpy())
            preds_300.extend(p300.cpu().numpy())
            orig_idxs.extend(orig_idx.numpy())
    return np.array(preds_60), np.array(preds_300), np.array(orig_idxs)


# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]
all_frames    = []

for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df_day = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded — check S3 paths and contracts")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d cols", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Targets
# ---------------------------------------------------------------------------

log.info("=== Step 3: Building targets ===")

df = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
for h in HORIZONS:
    df[f"ret_{h}s"] = (g["log_mid"].shift(-h) - df["log_mid"]) * 1e4

for h in HORIZONS:
    pct = df[f"ret_{h}s"].notna().mean() * 100
    log.info("Target ret_%ds: %.1f%% non-null", h, pct)

# ---------------------------------------------------------------------------
# Step 4 — Train/val/test split
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting ===")

all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]

log.info(
    "Train: %d days  Val: %d days  Test: %d days",
    len(train_dates), len(val_dates), len(test_dates),
)

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

df_train = df[mask_train].copy()
df_val   = df[mask_val].copy()
df_test  = df[mask_test].copy()

# ---------------------------------------------------------------------------
# Step 5 — Build sequence datasets
# ---------------------------------------------------------------------------

log.info("=== Step 5: Building sequence datasets ===")

# Check all features present
missing = [f for f in FEATURE_SET if f not in df.columns]
if missing:
    raise RuntimeError(f"Missing features: {missing}")

# Replace NaN in features with 0 (forward-fill already done in builder;
# remaining NaN at day-start of rolling windows → zero-pad)
df[FEATURE_SET] = df[FEATURE_SET].fillna(0.0)

ds_train = SequenceDataset(
    df_train, FEATURE_SET, SEQ_LEN, fit_scaler=True
)
ds_val = SequenceDataset(
    df_val, FEATURE_SET, SEQ_LEN,
    scaler=ds_train.scaler, fit_scaler=False
)
ds_test = SequenceDataset(
    df_test, FEATURE_SET, SEQ_LEN,
    scaler=ds_train.scaler, fit_scaler=False
)

log.info(
    "Samples — train: %d  val: %d  test: %d",
    len(ds_train), len(ds_val), len(ds_test),
)

loader_train = DataLoader(
    ds_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False
)
loader_val   = DataLoader(
    ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False
)
loader_test  = DataLoader(
    ds_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False
)

# ---------------------------------------------------------------------------
# Step 6 — Build and train TCN
# ---------------------------------------------------------------------------

log.info("=== Step 6: Training TCN ===")

model = TCN(
    n_features=len(FEATURE_SET),
    n_channels=TCN_CHANNELS,
    n_layers=TCN_LAYERS,
    kernel=KERNEL_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log.info("TCN parameters: %d", n_params)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=LR * 0.1
)
criterion = nn.HuberLoss(delta=1.0)   # robust to outliers vs MSE

best_val_ic = -np.inf
best_epoch  = 0
best_state  = None
train_history = []

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_ic60, tr_ic300 = run_epoch(
        model, loader_train, optimizer, criterion, DEVICE, train=True
    )
    va_loss, va_ic60, va_ic300 = run_epoch(
        model, loader_val, optimizer, criterion, DEVICE, train=False
    )
    scheduler.step()

    # Combined IC as stopping criterion (average of both horizons)
    val_ic_combined = (va_ic60 + va_ic300) / 2.0

    train_history.append({
        "epoch":      epoch,
        "tr_loss":    round(tr_loss, 6),
        "tr_ic60":    round(tr_ic60,  4),
        "tr_ic300":   round(tr_ic300, 4),
        "va_loss":    round(va_loss,  6),
        "va_ic60":    round(va_ic60,  4),
        "va_ic300":   round(va_ic300, 4),
    })

    if epoch % 5 == 0 or epoch == 1:
        log.info(
            "Ep%03d | tr_loss=%.5f tr_ic60=%.4f tr_ic300=%.4f | "
            "va_ic60=%.4f va_ic300=%.4f",
            epoch, tr_loss, tr_ic60, tr_ic300, va_ic60, va_ic300,
        )

    if val_ic_combined > best_val_ic:
        best_val_ic = val_ic_combined
        best_epoch  = epoch
        best_state  = {k: v.clone() for k, v in model.state_dict().items()}

    # Early stopping
    if epoch - best_epoch >= PATIENCE:
        log.info("Early stopping at epoch %d (best epoch %d)", epoch, best_epoch)
        break

log.info(
    "Best epoch: %d  best val IC (avg): %.4f", best_epoch, best_val_ic
)

# Restore best weights
model.load_state_dict(best_state)

# Save model
torch.save(
    {
        "model_state": best_state,
        "scaler":      ds_train.scaler,
        "features":    FEATURE_SET,
        "seq_len":     SEQ_LEN,
        "config": {
            "n_channels": TCN_CHANNELS,
            "n_layers":   TCN_LAYERS,
            "kernel":     KERNEL_SIZE,
            "dropout":    DROPOUT,
        },
        "best_epoch":  best_epoch,
        "train_dates": [str(d) for d in train_dates],
    },
    MODEL_DIR / "tcn_usdinr.pt",
)
log.info("Model saved → models/tcn_usdinr.pt")

# ---------------------------------------------------------------------------
# Step 7 — Evaluate on all splits
# ---------------------------------------------------------------------------

log.info("=== Step 7: Evaluating ===")


def eval_split(loader, split_name):
    p60, p300, idxs = predict_all(model, loader, DEVICE)
    rows = []
    for orig_idx, pp60, pp300 in zip(idxs, p60, p300):
        rows.append({"orig_idx": int(orig_idx), "pred_60s": pp60, "pred_300s": pp300})
    pred_df = pd.DataFrame(rows).set_index("orig_idx")
    # Join back to actual targets
    actual = df.loc[pred_df.index, ["ret_60s", "ret_300s", "date", "ts_sec", "symbol"]]
    pred_df = pred_df.join(actual)

    results = {}
    for h, pc, ac in [(60, "pred_60s", "ret_60s"), (300, "pred_300s", "ret_300s")]:
        valid = pred_df[[pc, ac]].dropna()
        ic  = compute_ic(valid[ac].values, valid[pc].values)
        rmse = float(np.sqrt(np.mean((valid[ac].values - valid[pc].values) ** 2)))
        hit  = float(np.mean(np.sign(valid[ac].values) == np.sign(valid[pc].values)))
        log.info(
            "%s/ret_%ds | IC=%+.4f RMSE=%.4f Hit=%.4f n=%d",
            split_name, h, ic, rmse, hit, len(valid),
        )
        results[f"ret_{h}s"] = {"ic": ic, "rmse": rmse, "hit_rate": hit, "n": len(valid)}
    return results, pred_df


tr_metrics, tr_pred = eval_split(loader_train, "train")
va_metrics, va_pred = eval_split(loader_val,   "val")
te_metrics, te_pred = eval_split(loader_test,  "test")

# ---------------------------------------------------------------------------
# Step 8 — Daily IC breakdown
# ---------------------------------------------------------------------------

log.info("=== Step 8: Daily IC ===")

daily_ic_rows = []

for split_name, pred_df, split_dates in [
    ("val",  va_pred, val_dates),
    ("test", te_pred, test_dates),
]:
    for day, grp in pred_df.groupby("date"):
        for h, pc, ac in [(60, "pred_60s", "ret_60s"), (300, "pred_300s", "ret_300s")]:
            valid = grp[[pc, ac]].dropna()
            if len(valid) < 50:
                continue
            ic = compute_ic(valid[ac].values, valid[pc].values)
            daily_ic_rows.append({
                "date":      str(day),
                "horizon_s": h,
                "ic":        round(ic, 4),
                "n":         len(valid),
                "split":     split_name,
            })

daily_ic = pd.DataFrame(daily_ic_rows)

# ---------------------------------------------------------------------------
# Step 9 — Comparison table (TCN vs LightGBM baseline)
# ---------------------------------------------------------------------------

log.info("=== Step 9: Building comparison ===")

# LightGBM results from lgbm_baseline.py (update these after running lgbm_baseline)
LGBM_RESULTS = {
    # (feat_set, horizon): test_ic  — paste from results/lgbm_baseline/comparison.csv
    ("base46",       60):  None,
    ("base46",      300):  None,
    ("challenger50",  60):  None,
    ("challenger50", 300):  None,
}

# Try to load lgbm comparison from disk automatically
lgbm_csv = pathlib.Path(r"C:\projects\hedge_fund\results\lgbm_baseline\comparison.csv")
if lgbm_csv.exists():
    lgbm_df = pd.read_csv(lgbm_csv)
    for _, row in lgbm_df.iterrows():
        LGBM_RESULTS[(row["feat_set"], int(row["horizon_s"]))] = float(row["test_ic"])
    log.info("Loaded LightGBM results from lgbm_baseline/comparison.csv")
else:
    log.warning("lgbm_baseline/comparison.csv not found — fill LGBM_RESULTS manually")

comparison_rows = []

# TCN rows
for h in HORIZONS:
    comparison_rows.append({
        "model":      "tcn_challenger50",
        "n_features": len(FEATURE_SET),
        "horizon_s":  h,
        "train_ic":   round(tr_metrics[f"ret_{h}s"]["ic"], 4),
        "val_ic":     round(va_metrics[f"ret_{h}s"]["ic"], 4),
        "test_ic":    round(te_metrics[f"ret_{h}s"]["ic"], 4),
        "test_rmse":  round(te_metrics[f"ret_{h}s"]["rmse"], 4),
        "test_hit":   round(te_metrics[f"ret_{h}s"]["hit_rate"], 4),
        "best_iter":  best_epoch,
    })

# LightGBM rows (from saved results)
for (feat_set, h), test_ic in LGBM_RESULTS.items():
    comparison_rows.append({
        "model":      f"lgbm_{feat_set}",
        "n_features": len(BASE46) if feat_set == "base46" else len(CHALLENGER50),
        "horizon_s":  h,
        "train_ic":   None,
        "val_ic":     None,
        "test_ic":    test_ic,
        "test_rmse":  None,
        "test_hit":   None,
        "best_iter":  None,
    })

comparison = pd.DataFrame(comparison_rows)

# ---------------------------------------------------------------------------
# Step 10 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 10: Saving outputs ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(
        {
            "split_dates": {
                "train": [str(d) for d in train_dates],
                "val":   [str(d) for d in val_dates],
                "test":  [str(d) for d in test_dates],
            },
            "config": {
                "seq_len":     SEQ_LEN,
                "feature_set": "CHALLENGER50",
                "n_features":  len(FEATURE_SET),
                "tcn_channels": TCN_CHANNELS,
                "tcn_layers":  TCN_LAYERS,
                "kernel_size": KERNEL_SIZE,
                "dropout":     DROPOUT,
                "batch_size":  BATCH_SIZE,
                "max_epochs":  MAX_EPOCHS,
                "best_epoch":  best_epoch,
                "device":      DEVICE,
            },
            "metrics": {
                "train": tr_metrics,
                "val":   va_metrics,
                "test":  te_metrics,
            },
            "train_history": train_history,
        },
        f,
        indent=2,
    )
log.info("Saved metrics.json")

comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
log.info("Saved comparison.csv")

if not daily_ic.empty:
    daily_ic.to_csv(OUT_DIR / "daily_ic.csv", index=False)
    log.info("Saved daily_ic.csv")

# OOF predictions
oof_parts = []
for split_name, pred_df in [("val", va_pred), ("test", te_pred)]:
    pred_df = pred_df.copy()
    pred_df["split"] = split_name
    oof_parts.append(pred_df)
oof_out = pd.concat(oof_parts)
oof_out.to_csv(OUT_DIR / "oof_predictions.csv")
log.info("Saved oof_predictions.csv (%d rows)", len(oof_out))

# ---------------------------------------------------------------------------
# Step 11 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("TCN  vs  LightGBM  —  COMPARISON SUMMARY")
print("=" * 70)
print(comparison.to_string(index=False))

print("\n--- TCN vs LightGBM CHALLENGER50 (test IC) ---")
for h in HORIZONS:
    lgbm_ic = LGBM_RESULTS.get(("challenger50", h))
    tcn_ic  = te_metrics[f"ret_{h}s"]["ic"]
    if lgbm_ic is not None:
        delta = tcn_ic - lgbm_ic
        print(f"  ret_{h}s:  LightGBM={lgbm_ic:+.4f}  TCN={tcn_ic:+.4f}  delta={delta:+.4f}")
    else:
        print(f"  ret_{h}s:  TCN={tcn_ic:+.4f}  (LightGBM baseline not loaded)")

if not daily_ic.empty:
    print("\n--- Daily IC (test days only) ---")
    print(
        daily_ic[daily_ic["split"] == "test"]
        .groupby("horizon_s")["ic"]
        .agg(["mean", "std", "min"])
        .round(4)
        .to_string()
    )
print("=" * 70)

12:09:57  INFO  Device: cpu
12:09:57  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (7 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

12:11:47  INFO  Raw: 472806 rows | symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT', 'USDINR26NOVFUT']
12:11:47  INFO  === Step 2: Building features ===
12:11:51  INFO  preprocess: 881382 rows after regularization
12:11:51  INFO  calculate_features: building CORE31 ...
12:12:13  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
12:12:14  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
12:12:20  INFO  _build_book_geometry_features: built 881382 rows
12:12:20  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
12:12:22  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
12:12:22  INFO  _build_shock_recovery_features: built 881382 rows
12:12:22  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
12:12:24  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
12:12:25  INFO  _build_update_clustering_features: built 881382 rows
12:12:25  INFO  


TCN  vs  LightGBM  —  COMPARISON SUMMARY
            model  n_features  horizon_s  train_ic  val_ic  test_ic  test_rmse  test_hit  best_iter
 tcn_challenger50          50         60    0.0259 -0.0005   0.0287     0.9185    0.3797        2.0
 tcn_challenger50          50        300    0.0188 -0.0248   0.0134     2.3012    0.4692        2.0
      lgbm_base46          46         60       NaN     NaN   0.2211        NaN       NaN        NaN
      lgbm_base46          46        300       NaN     NaN   0.1089        NaN       NaN        NaN
lgbm_challenger50          50         60       NaN     NaN   0.2322        NaN       NaN        NaN
lgbm_challenger50          50        300       NaN     NaN   0.1054        NaN       NaN        NaN

--- TCN vs LightGBM CHALLENGER50 (test IC) ---
  ret_60s:  LightGBM=+0.2322  TCN=+0.0287  delta=-0.2035
  ret_300s:  LightGBM=+0.1054  TCN=+0.0134  delta=-0.0920

--- Daily IC (test days only) ---
             mean  std     min
horizon_s                    